In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import matplotlib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import lightgbm as lgb
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

font_path = "C:/Windows/Fonts/gulim.ttc"
font = fm.FontProperties(fname=font_path).get_name()
matplotlib.rc("font", family=font)

In [2]:
df = pd.read_csv("../../EDA/Merge/data/result data/Final DF.csv")

In [3]:
df

,Year,Nation,Eng_Nation,Wc_Rank,Wc_Point,Q_WR,Q_GR,F_Rank,F_Point,F_Rd,...,FS_6,FS_7,FS_8,FS_9,FS_10,FS_11,FS_12,FS_13,ATK_INDEX,DEF_INDEX
0,2002,브라질,Brazil,1,21,0.46,2.352113,1.33,818.33,0.00,...,67.95,75.27,77.05,72.73,82.09,31.95,68.95,59.61,0.141500,-0.070093
1,2002,독일,Germany,2,16,0.55,1.817073,8.67,718.33,0.00,...,63.29,66.38,67.86,72.57,80.57,37.60,62.88,63.29,0.031444,0.011481
2,2002,터키,Turkey,3,13,0.46,1.457627,33.00,593.33,0.00,...,62.65,66.45,69.80,67.05,74.05,30.85,65.18,56.78,-0.039941,0.015588
3,2002,대한민국,South Korea,4,11,0.48,2.216667,42.00,573.00,0.00,...,55.41,65.91,65.50,65.68,72.45,32.64,57.66,62.23,0.040250,0.133021
4,2002,스페인,Spain,5,11,0.65,3.227273,5.00,742.33,0.00,...,69.77,75.50,78.23,78.09,83.27,35.05,69.80,64.55,-0.105762,-0.099683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,2022,덴마크,Denmark,28,1,0.61,2.446429,13.33,1611.83,-18.00,...,68.77,72.19,70.69,67.92,70.15,17.13,66.37,53.93,NaN,NaN
188,2022,세르비아,Serbia,29,1,0.48,1.536232,31.33,1492.66,-18.00,...,62.65,66.19,72.58,69.38,64.15,18.21,64.19,51.35,NaN,NaN
189,2022,웨일스,Wales,30,1,0.44,1.233333,21.00,1545.20,10.75,...,67.38,66.19,63.88,55.19,66.27,17.62,59.62,53.12,NaN,NaN
190,2022,캐나다,Canada,31,0,0.48,2.296875,66.33,1356.74,-34.00,...,66.48,66.48,69.76,60.36,65.60,16.50,56.82,43.71,NaN,NaN


In [4]:
df.columns

Index(['Year', 'Nation', 'Eng_Nation', 'Wc_Rank', 'Wc_Point', 'Q_WR', 'Q_GR',
       'F_Rank', 'F_Point', 'F_Rd', 'F_Pd', 'Avg_Apps', 'Avg_Age',
       'Avg_Famous', 'FS_0', 'FS_1', 'FS_2', 'FS_3', 'FS_4', 'FS_5', 'FS_6',
       'FS_7', 'FS_8', 'FS_9', 'FS_10', 'FS_11', 'FS_12', 'FS_13', 'ATK_INDEX',
       'DEF_INDEX'],
      dtype='object')

In [5]:
# 그룹핑 함수
def rank_stage(rank):
    if rank <= 2:
        return f"{rank}위"
    elif rank <= 4:
        return "4강"
    elif rank <= 8:
        return "8강"
    elif rank <= 16:
        return "16강"
    else:
        return "조별리그"


# 새로운 열 생성
df["Rank_Group"] = df["Wc_Rank"].apply(rank_stage)

In [6]:
# Year, Nation 기준 정렬
df = df.sort_values(["Nation", "Year"])

# 직전 월드컵 순위 구하기
df["PrevRank"] = df.groupby("Nation")["Wc_Rank"].shift(1)

# NaN 은 33등으로 가정
df["PrevRank"] = df["PrevRank"].fillna(33).astype(int)

In [7]:
df.columns

Index(['Year', 'Nation', 'Eng_Nation', 'Wc_Rank', 'Wc_Point', 'Q_WR', 'Q_GR',
       'F_Rank', 'F_Point', 'F_Rd', 'F_Pd', 'Avg_Apps', 'Avg_Age',
       'Avg_Famous', 'FS_0', 'FS_1', 'FS_2', 'FS_3', 'FS_4', 'FS_5', 'FS_6',
       'FS_7', 'FS_8', 'FS_9', 'FS_10', 'FS_11', 'FS_12', 'FS_13', 'ATK_INDEX',
       'DEF_INDEX', 'Rank_Group', 'PrevRank'],
      dtype='object')

In [8]:
df = df.sort_values(["Nation", "Year"])

# 직전 대회 성적
df["PrevRank"] = df.groupby("Nation")["Wc_Rank"].shift(1)
df["PrevRank"] = df["PrevRank"].fillna(33).astype(int)

# 등수 변화 계산 (양수면 내려간 것, 음수면 올라간 것)
df["RankChange"] = df["Wc_Rank"] - df["PrevRank"]

print(df)

     Year Nation Eng_Nation  Wc_Rank  Wc_Point  Q_WR      Q_GR  F_Rank  \
44   2006     가나      Ghana       13         6  0.48  1.590164   70.33   
70   2010     가나      Ghana        7         8  0.46  1.558824   32.33   
120  2014     가나      Ghana       25         1  0.44  1.910448   30.67   
183  2022     가나      Ghana       24         3  0.41  1.148148   50.00   
88   2010    그리스     Greece       25         3  0.60  1.417910   15.00   
..    ...    ...        ...      ...       ...   ...       ...     ...   
47   2006     호주  Australia       16         4  0.54  2.604651   57.00   
84   2010     호주  Australia       21         4  0.56  2.236364   33.67   
125  2014     호주  Australia       30         0  0.51  1.973684   32.33   
157  2018     호주  Australia       30         1  0.60  2.604651   52.00   
170  2022     호주  Australia       11         6  0.49  2.970588   39.00   

     F_Point   F_Rd  ...   FS_9  FS_10  FS_11  FS_12  FS_13  ATK_INDEX  \
44    531.33  18.00  ...  63.45  76.7

In [9]:
df.columns

Index(['Year', 'Nation', 'Eng_Nation', 'Wc_Rank', 'Wc_Point', 'Q_WR', 'Q_GR',
       'F_Rank', 'F_Point', 'F_Rd', 'F_Pd', 'Avg_Apps', 'Avg_Age',
       'Avg_Famous', 'FS_0', 'FS_1', 'FS_2', 'FS_3', 'FS_4', 'FS_5', 'FS_6',
       'FS_7', 'FS_8', 'FS_9', 'FS_10', 'FS_11', 'FS_12', 'FS_13', 'ATK_INDEX',
       'DEF_INDEX', 'Rank_Group', 'PrevRank', 'RankChange'],
      dtype='object')

In [10]:
# 숫자형 컬럼 추출 (Year, Nation 등 제외)
exclude_cols = ["Year", "Nation", "Eng_Nation","Wc_Rank","Wc_Point","PrevRank","RankChange"]  # 그대로 두고 싶은 컬럼
numeric_cols = df.select_dtypes(include=[np.number]).columns.difference(exclude_cols)

# 연도별 표준화
df_std = df.copy()
df_std[numeric_cols] = df.groupby("Year")[numeric_cols].transform(
    lambda x: (x - x.mean())
)

In [11]:
df_std

,Year,Nation,Eng_Nation,Wc_Rank,Wc_Point,Q_WR,Q_GR,F_Rank,F_Point,F_Rd,...,FS_9,FS_10,FS_11,FS_12,FS_13,ATK_INDEX,DEF_INDEX,Rank_Group,PrevRank,RankChange
44,2006,가나,Ghana,13,6,-0.040625,-0.392364,41.350625,-135.076250,18.552187,...,-2.172500,2.986563,-5.474375,-4.038125,-1.095000,-0.027566,-0.001682,16강,33,-20
70,2010,가나,Ghana,7,8,-0.047187,-0.277379,-0.554688,-78.579687,-39.124688,...,-1.074688,-1.056562,-5.125938,-8.126562,-7.218750,-0.007826,-0.009683,8강,13,-6
120,2014,가나,Ghana,25,1,-0.086250,-0.119724,7.992812,-170.447813,2.901875,...,0.335625,5.251250,-4.411562,3.411875,2.279375,0.165043,0.032367,조별리그,7,18
183,2022,가나,Ghana,24,3,-0.122188,-1.202422,24.510625,-126.280625,14.993750,...,-4.780000,1.839375,-2.860000,-4.245313,-1.668438,NaN,NaN,조별리그,25,-1
88,2010,그리스,Greece,25,3,0.092812,-0.418292,-17.884687,91.750313,-6.124688,...,0.845312,3.163438,0.004062,3.343438,0.281250,-0.166367,-0.127894,조별리그,33,-8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47,2006,호주,Australia,16,4,0.019375,0.622124,28.020625,-108.406250,-3.117813,...,-4.352500,-0.713437,3.075625,-0.888125,-1.525000,0.204002,0.207539,16강,33,-17
84,2010,호주,Australia,21,4,0.052813,0.400161,0.785313,-124.919687,-24.454687,...,-1.114688,-2.896563,0.974062,-0.186562,2.861250,-0.027575,-0.109819,조별리그,16,5
125,2014,호주,Australia,30,0,-0.016250,-0.056488,9.652812,-159.117813,3.221875,...,-3.294375,-3.298750,0.908438,-2.228125,-3.190625,-0.042722,-0.129496,조별리그,21,9
157,2018,호주,Australia,30,1,0.080937,0.479957,22.992812,-287.439375,23.246563,...,-2.576875,-1.425625,1.379687,-5.081875,-4.900625,0.090695,0.028222,조별리그,30,0


In [12]:
X = df_std.drop(
    [
        "Year",
        "Nation",
        "Eng_Nation",
        "Rank_Group",
        "RankChange",
        "ATK_INDEX",
        "DEF_INDEX",
        "Wc_Rank",
        "Wc_Point",
        "FS_0",
        "FS_1",
        "FS_2",
        "FS_3",
        "FS_4",
        "FS_5",
        "FS_6",
        "FS_7",
        "FS_8",
        "FS_9",
        "FS_10",
        "FS_11",
        "FS_12",
        "FS_13",
    ],
    axis=1,
)
y = df_std["Rank_Group"]



le = LabelEncoder()
y_encoded = le.fit_transform(y)
model = XGBClassifier(
    n_estimators=100, learning_rate=0.01, max_depth=1, verbose=-1
)

cross_val_score(model,X,y_encoded,cv=5).mean()

c:\Users\chris\anaconda3\envs\ml_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:06:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\chris\anaconda3\envs\ml_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:06:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\chris\anaconda3\envs\ml_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:06:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\chris\anaconda3\envs\ml_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:06:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "verbose" } are not used.

  bst.up

np.float64(0.49014844804318497)

In [13]:
y

44      16강
70       8강
120    조별리그
183    조별리그
88     조별리그
       ... 
47      16강
84     조별리그
125    조별리그
157    조별리그
170     16강
Name: Rank_Group, Length: 192, dtype: object

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import lightgbm as lgb
from sklearn.model_selection import cross_val_score

model = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.01, max_depth=5, random_state=42, verbose=-1
)

cross_val_score(model, X, y, cv=10).mean()

c:\Users\chris\anaconda3\envs\ml_env\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  warnings.warn(


np.float64(0.40131578947368424)

In [15]:
import pandas as pd
import numpy as np

numeric_cols = [
    "Q_WR",
    "Q_GR",
    "F_Rank",
    "F_Point",
    "F_Rd",
    "F_Pd",
    "Avg_Apps",
    "Avg_Age",
    "Avg_Famous",
    "PrevRank",
]


def augment_with_noise(df, numeric_cols, n_aug=2, noise_level=0.05):
    augmented = []
    for _ in range(n_aug):
        df_aug = df.copy()
        df_aug[numeric_cols] = df_aug[numeric_cols] * (
            1 + noise_level * np.random.randn(len(df), len(numeric_cols))
        )
        augmented.append(df_aug)
    return pd.concat([df] + augmented, ignore_index=True)


df_aug = augment_with_noise(df, numeric_cols)
print("증강 후 데이터 수:", len(df_aug))

증강 후 데이터 수: 576


In [16]:
# 예: 비율, 차이, 평균
df_aug["F_Point_per_Age"] = df_aug["F_Point"] / df_aug["Avg_Age"]
df_aug["F_Point_per_App"] = df_aug["F_Point"] / (df_aug["Avg_Apps"] + 1e-3)
df_aug["Rank_Diff"] = df_aug["PrevRank"] - df_aug["F_Rank"]
df_aug["Famous_per_Age"] = df_aug["Avg_Famous"] / df_aug["Avg_Age"]

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import lightgbm as lgb

numeric_cols = [
    "Q_WR",
    "Q_GR",
    "F_Rank",
    "F_Point",
    "F_Rd",
    "F_Pd",
    "Avg_Apps",
    "Avg_Age",
    "Avg_Famous",
    "PrevRank",
]


# ---------------------------
# 2️⃣ 데이터 증강 (Noise 추가)
# ---------------------------
def augment_with_noise(df, numeric_cols, n_aug=2, noise_level=0.05):
    augmented = []
    for _ in range(n_aug):
        df_aug = df.copy()
        df_aug[numeric_cols] = df_aug[numeric_cols] * (
            1 + noise_level * np.random.randn(len(df), len(numeric_cols))
        )
        augmented.append(df_aug)
    return pd.concat([df] + augmented, ignore_index=True)


df_aug = augment_with_noise(df, numeric_cols)

# ---------------------------
# 3️⃣ Feature Engineering
# ---------------------------
df_aug["F_Point_per_Age"] = df_aug["F_Point"] / df_aug["Avg_Age"]
df_aug["F_Point_per_App"] = df_aug["F_Point"] / (df_aug["Avg_Apps"] + 1e-3)
df_aug["Rank_Diff"] = df_aug["PrevRank"] - df_aug["F_Rank"]
df_aug["Famous_per_Age"] = df_aug["Avg_Famous"] / df_aug["Avg_Age"]

features = numeric_cols + [
    "F_Point_per_Age",
    "F_Point_per_App",
    "Rank_Diff",
    "Famous_per_Age",
]

X = df_aug[features]
y = df_aug["Rank_Group"]

# ---------------------------
# 4️⃣ 학습용/검증용 분리
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------
# 5️⃣ LightGBM 모델 학습 (callbacks 사용)
# ---------------------------
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=10,  # 깊이 늘리기
    min_child_samples=1,  # 최소 샘플 수 줄이기
    min_child_weight=0.001,  # 최소 가중치 줄이기
    random_state=42,
)


model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)],
)

# ---------------------------
# 6️⃣ 모델 성능 측정
# ---------------------------
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=df["Rank_Group"].unique())
report = classification_report(y_test, y_pred)

print("Accuracy:", acc)
print("Confusion Matrix:\n", cm)
print("Classification Report:\n", report)

Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 0.839336
[100]	valid_0's multi_logloss: 0.664852
[150]	valid_0's multi_logloss: 0.581363
[200]	valid_0's multi_logloss: 0.537311
[250]	valid_0's multi_logloss: 0.489152
[300]	valid_0's multi_logloss: 0.459635
[350]	valid_0's multi_logloss: 0.460622
Early stopping, best iteration is:
[320]	valid_0's multi_logloss: 0.456924
Accuracy: 0.8793103448275862
Confusion Matrix:
 [[27  0  2  0  0  0]
 [ 0 11  3  0  0  0]
 [ 2  0 56  0  0  0]
 [ 1  0  0  3  0  0]
 [ 0  0  5  0  2  0]
 [ 0  0  1  0  0  3]]
Classification Report:
               precision    recall  f1-score   support

         16강       0.90      0.93      0.92        29
          1위       1.00      0.75      0.86         4
          2위       1.00      0.75      0.86         4
          4강       1.00      0.29      0.44         7
          8강       1.00      0.79      0.88        14
        조별리그       0.84      0.97      0.90        58

    ac

In [18]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score

numeric_cols = [
    "Q_WR",
    "Q_GR",
    "F_Rank",
    "F_Point",
    "F_Rd",
    "F_Pd",
    "Avg_Apps",
    "Avg_Age",
    "Avg_Famous",
    "PrevRank",
]

# ---------------------------
# 1️⃣ 각 Rank_Group에서 소수 샘플을 test로 분리
# ---------------------------
test_indices = []
for grp in df["Rank_Group"].unique():
    grp_idx = df[df["Rank_Group"] == grp].index
    test_indices.extend(np.random.choice(grp_idx, 2, replace=False))  # 각 그룹 2개씩

test_df = df.loc[test_indices].reset_index(drop=True)
train_df = df.drop(test_indices).reset_index(drop=True)


# ---------------------------
# 2️⃣ Feature Engineering (train/test 모두 동일하게)
# ---------------------------
def add_features(df):
    df = df.copy()
    df["F_Point_per_Age"] = df["F_Point"] / df["Avg_Age"]
    df["F_Point_per_App"] = df["F_Point"] / (df["Avg_Apps"] + 1e-3)
    df["Rank_Diff"] = df["PrevRank"] - df["F_Rank"]
    df["Famous_per_Age"] = df["Avg_Famous"] / df["Avg_Age"]
    return df


train_df = add_features(train_df)
test_df = add_features(test_df)

features = numeric_cols + [
    "F_Point_per_Age",
    "F_Point_per_App",
    "Rank_Diff",
    "Famous_per_Age",
]

X_train = train_df[features]
y_train = train_df["Rank_Group"]
X_test = test_df[features]
y_test = test_df["Rank_Group"]

# ---------------------------
# 3️⃣ LightGBM 학습 + 교차검증
# ---------------------------
model = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.01, max_depth=5, random_state=42, verbose=-1
)

# 교차검증
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
print("CV Accuracy:", cv_scores.mean())

# 최종 학습 및 평가
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

c:\Users\chris\anaconda3\envs\ml_env\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


CV Accuracy: 0.40555555555555556


In [19]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score

numeric_cols = [
    "Q_WR",
    "Q_GR",
    "F_Rank",
    "F_Point",
    "F_Rd",
    "F_Pd",
    "Avg_Apps",
    "Avg_Age",
    "Avg_Famous",
    "PrevRank",
]

# ---------------------------
# 1️⃣ 각 Rank_Group에서 소수 샘플을 test로 분리
# ---------------------------
test_indices = []
for grp in df["Rank_Group"].unique():
    grp_idx = df[df["Rank_Group"] == grp].index
    test_indices.extend(np.random.choice(grp_idx, 2, replace=False))  # 각 그룹 2개씩

test_df = df.loc[test_indices].reset_index(drop=True)
train_df = df.drop(test_indices).reset_index(drop=True)

# ---------------------------
# 2️⃣ Train Feature Engineering
# ---------------------------
train_df["F_Point_per_Age"] = train_df["F_Point"] / train_df["Avg_Age"]
train_df["F_Point_per_App"] = train_df["F_Point"] / (train_df["Avg_Apps"] + 1e-3)
train_df["Rank_Diff"] = train_df["PrevRank"] - train_df["F_Rank"]
train_df["Famous_per_Age"] = train_df["Avg_Famous"] / train_df["Avg_Age"]

features = numeric_cols + [
    "F_Point_per_Age",
    "F_Point_per_App",
    "Rank_Diff",
    "Famous_per_Age",
]
X_train = train_df[features]
y_train = train_df["Rank_Group"]


# ---------------------------
# 3️⃣ Test 증강 (Noise)
# ---------------------------
def augment_with_noise(df, numeric_cols, n_aug=50, noise_level=0.03):
    augmented = []
    for _ in range(n_aug):
        df_aug = df.copy()
        df_aug[numeric_cols] = df_aug[numeric_cols] * (
            1 + noise_level * np.random.randn(len(df), len(numeric_cols))
        )
        augmented.append(df_aug)
    return pd.concat(augmented, ignore_index=True)


test_aug = augment_with_noise(test_df, numeric_cols, n_aug=50, noise_level=0.03)

# Feature engineering on test_aug
test_aug["F_Point_per_Age"] = test_aug["F_Point"] / test_aug["Avg_Age"]
test_aug["F_Point_per_App"] = test_aug["F_Point"] / (test_aug["Avg_Apps"] + 1e-3)
test_aug["Rank_Diff"] = test_aug["PrevRank"] - test_aug["F_Rank"]
test_aug["Famous_per_Age"] = test_aug["Avg_Famous"] / test_aug["Avg_Age"]

X_test = test_aug[features]
y_test = test_aug["Rank_Group"]

# ---------------------------
# 4️⃣ LightGBM 학습
# ---------------------------
model = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.01, max_depth=5, random_state=42, verbose=-1
)

cross_val_score(model, X, y)

array([0.92241379, 0.88695652, 0.92173913, 0.93913043, 0.86956522])

In [22]:
import pandas as pd

# ==========================
# 1. 데이터 로드
# ==========================
# 팀 특징 데이터 (연도별 국가별 특징)
team_features = pd.read_csv("../../EDA/Merge/data/result data/Final DF.csv")
# 예시 컬럼: [Year, Nation, Eng_Nation, Wc_Rank, Wc_Point, ..., ATK_INDEX, DEF_INDEX]

# 매치 데이터
matches = pd.read_csv(
    "../../EDA/Qualifier Match/data/source data/results.csv", parse_dates=["date"]
)
matches["year"] = matches["date"].dt.year

In [23]:
team_features.columns

Index(['Year', 'Nation', 'Eng_Nation', 'Wc_Rank', 'Wc_Point', 'Q_WR', 'Q_GR',
       'F_Rank', 'F_Point', 'F_Rd', 'F_Pd', 'Avg_Apps', 'Avg_Age',
       'Avg_Famous', 'FS_0', 'FS_1', 'FS_2', 'FS_3', 'FS_4', 'FS_5', 'FS_6',
       'FS_7', 'FS_8', 'FS_9', 'FS_10', 'FS_11', 'FS_12', 'FS_13', 'ATK_INDEX',
       'DEF_INDEX'],
      dtype='object')

In [ ]:
import pandas as pd

# 표준화할 컬럼 목록 (수치형만 선택)
cols_to_scale = [
    'Wc_Rank', 'Wc_Point', 'Q_WR', 'Q_GR', 'F_Rank', 'F_Point',
    'F_Rd', 'F_Pd', 'Avg_Apps', 'Avg_Age', 'Avg_Famous',
    'FS_0','FS_1','FS_2','FS_3','FS_4','FS_5','FS_6','FS_7','FS_8','FS_9',
    'FS_10','FS_11','FS_12','FS_13'
]

# Year 기준 그룹화하여 표준화
df_scaled = team_features.copy()
df_scaled[cols_to_scale] = df_scaled.groupby('Year')[cols_to_scale].transform(
    lambda x: (x - x.mean()) / x.std()
)

print(df_scaled.head())


   Year Nation   Eng_Nation   Wc_Rank  Wc_Point      Q_WR      Q_GR    F_Rank  \
0  2002    브라질       Brazil -1.652306  3.266516 -0.516338  0.703468 -1.339299   
1  2002     독일      Germany -1.545705  2.212801  0.776753 -0.168670 -0.968084   
2  2002     터키       Turkey -1.439105  1.580572 -0.516338 -0.754583  0.262385   
3  2002   대한민국  South Korea -1.332504  1.159086 -0.228985  0.482685  0.717553   
4  2002    스페인        Spain -1.225904  1.159086  2.213520  2.130018 -1.153692   

    F_Point  F_Rd  ...      FS_6      FS_7      FS_8      FS_9     FS_10  \
0  2.178276   NaN  ...  1.739421  1.257070  1.233421  0.786176  1.125489   
1  0.997689   NaN  ...  0.855755 -0.403072 -0.385885  0.756786  0.858107   
2 -0.478046   NaN  ...  0.734393 -0.390000 -0.044051 -0.257160 -0.288821   
3 -0.718059   NaN  ... -0.638513 -0.490842 -0.801724 -0.508810 -0.570275   
4  1.281030   NaN  ...  2.084544  1.300021  1.441341  1.770733  1.333061   

      FS_11     FS_12     FS_13  ATK_INDEX  DEF_INDEX  
